In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import docling
from pathlib import Path

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling.loader import DoclingLoader, ExportType

from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Get the file path to the PDF file
FILE_PATH = Path.cwd() / "RegulationsfortheFIFAWorldCup26_noimages.pdf"

# Set your pipeline options for docling
pipeline_options = PdfPipelineOptions(
    allow_external_plugins=True,
    do_ocr=False,
    do_table_structure=True,
    force_backend_text=True,
    generate_parsed_pages=False,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

loader = DoclingLoader(
    file_path=str(FILE_PATH),
    converter=converter,
    export_type=ExportType.MARKDOWN,
)

docs = loader.load()
markdown = docs[0].page_content

print("documents loaded:", len(docs))
print("markdown length:", len(markdown))

In [ ]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [("#", "Header 1"), 
                                                                ("##", "Header 2")])
pages_md_split = md_splitter.split_text(docs[0].page_content)                                                                

In [ ]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=pages_md_split,
    embedding=embedding,
)

print("chunks stored:", len(pages_md_split))


In [ ]:
from langchain_core.documents import Document

# Add duplicate docs to the vectorstore
new_doc_1 = Document(
    page_content=(
        "There are 48 teams in the world cup"
    ),
    metadata={"Header 2": "Number of teams in the world cup", "source": "manual"},
)

new_doc_2 = Document(
    page_content=(
        "There are 48 teams in the world cup"
    ),
    metadata={"Header 2": "Number of teams in the world cup", "source": "manual"},
)

new_doc_id_1 = vectorstore.add_documents([new_doc_1])
new_doc_id_2 = vectorstore.add_documents([new_doc_2])


In [ ]:
query = "What number of teams are in the FIFA World Cup 26?"
results = vectorstore.max_marginal_relevance_search(query, k=3, lambda_mult = 0.01)

for i, doc in enumerate(results):
    print(f"--- Result {i + 1} ---")
    print(doc.metadata)
    print(doc.page_content[:1000])
    print()

In [ ]:
query = "What number of teams are in the FIFA World Cup 26?"
results = vectorstore.max_marginal_relevance_search(query, k=10, lambda_mult = 0.01, filter = {'Header 2': 'Number of teams in the world cup'})

for i, doc in enumerate(results):
    print(f"--- Result {i + 1} ---")
    print(doc.metadata)
    print(doc.page_content[:1000])
    print()